In [ ]:
!pip install -q openai tqdm

import os
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from google.colab import userdata
from openai import OpenAI

# ----------------------------------------------------------------------
# 1. API & Model Configuration
# ----------------------------------------------------------------------
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Key 'OPENROUTER_API_KEY' not found in Colab Secrets.") from e

client = OpenAI(
    base_url="https://openrouter.ai/ai/v1" if False else "https://openrouter.ai/api/v1",
    api_key=api_key
)

MODEL_NAME = "x-ai/grok-4.3"

# ----------------------------------------------------------------------
# 2. Prompts Setup (Clear & Detailed for High Quality)
# ----------------------------------------------------------------------
SYSTEM_PROMPT = (
    "You are a strict binary evaluator. Return ONLY the single digit 1 or 0 as your final output. "
    "Do not output reasoning, commentary, or formatting."
)

USER_PROMPT_TEMPLATE = """Sentence (Ground Truth): {text}
Target Medicine: {target_medicine}

Task:
Determine if the target_medicine is correctly present and matched in the sentence.
- Output 1 if target_medicine is correct and present in the sentence.
- Output 0 if target_medicine is hallucinated.

Output strictly 1 or 0:"""

# ----------------------------------------------------------------------
# 3. Optimized & Safe Evaluation Function
# ----------------------------------------------------------------------
def evaluate_pair(text, target_medicine, max_retries=3):
    """
    Evaluates pair with 'low' reasoning effort for maximum quality
    and max_tokens=3 for minimum generation cost.
    """
    if pd.isna(text) or pd.isna(target_medicine) or not str(text).strip() or not str(target_medicine).strip():
        return 0

    user_prompt = USER_PROMPT_TEMPLATE.format(
        text=str(text).strip(),
        target_medicine=str(target_medicine).strip()
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0,
                max_tokens=3,  # Caps completion tokens for minimal cost
                extra_body={
                    "reasoning": {"effort": "low"}  # Low effort guarantees high accuracy without high token usage
                }
            )

            if response and response.choices and len(response.choices) > 0:
                choice = response.choices[0]
                if choice.message and choice.message.content is not None:
                    content = choice.message.content.strip()
                    match = re.search(r'[01]', content)
                    if match:
                        return int(match.group(0))

            time.sleep(1 * (attempt + 1))

        except Exception as e:
            if attempt == max_retries - 1:
                print(f"\n[API Error]: {e}")
            time.sleep(1 * (attempt + 1))

    return 0

# ----------------------------------------------------------------------
# 4. Data Loading & Resumable Checkpoint Setup
# ----------------------------------------------------------------------
input_file = "transformed_with_luna_modified_cleaned.csv"
output_file = "evaluated_grok_4_3_full_results.csv"

if os.path.exists(output_file):
    print(f"Resuming from existing output file '{output_file}'...")
    df = pd.read_csv(output_file)
else:
    df = pd.read_csv(input_file)

if 'correct_pair' not in df.columns:
    df['correct_pair'] = None
if 'incorrect_pair' not in df.columns:
    df['incorrect_pair'] = None

df['correct_pair'] = df['correct_pair'].astype(object)
df['incorrect_pair'] = df['incorrect_pair'].astype(object)

print(f"Total dataset size: {len(df)} rows.")

# ----------------------------------------------------------------------
# 5. Parallel Execution
# ----------------------------------------------------------------------
MAX_WORKERS = 5
SAVE_EVERY = 50

def process_row(idx, row):
    res_correct = row['correct_pair']
    res_incorrect = row['incorrect_pair']

    if pd.isna(res_correct) or str(res_correct).strip() == "":
        res_correct = evaluate_pair(row['text'], row['Medicine'])

    if pd.isna(res_incorrect) or str(res_incorrect).strip() == "":
        res_incorrect = evaluate_pair(row['modified_text'], row['Medicine'])

    return idx, int(res_correct), int(res_incorrect)

print(f"Evaluating rows using {MODEL_NAME} (Reasoning: Low) via OpenRouter...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_row, idx, row) for idx, row in df.iterrows()]

    completed_count = 0
    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, res_correct, res_incorrect = future.result()
        df.at[idx, 'correct_pair'] = res_correct
        df.at[idx, 'incorrect_pair'] = res_incorrect

        completed_count += 1
        if completed_count % SAVE_EVERY == 0:
            df.to_csv(output_file, index=False, encoding='utf-8-sig')

df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ Full evaluation finished! Saved results to '{output_file}'.")

# ----------------------------------------------------------------------
# 6. Accuracy Summary
# ----------------------------------------------------------------------
total_rows = len(df)
correct_pair_acc = (df['correct_pair'] == 1).sum() / total_rows * 100
incorrect_pair_acc = (df['incorrect_pair'] == 0).sum() / total_rows * 100

print("\n" + "="*45)
print("     GROK 4.3 FULL EVALUATION ACCURACY     ")
print("="*45)
print(f"Model Evaluated              : {MODEL_NAME}")
print(f"Total Evaluated Rows        : {total_rows}")
print(f"Correct Pair Accuracy (1s)   : {correct_pair_acc:.2f}%")
print(f"Incorrect Pair Accuracy (0s) : {incorrect_pair_acc:.2f}%")
print("="*45)